# Kaggle 2×T4 — Exact FP32 Downstream-Suffix JVP Audit (v2)

Use **Kaggle GPU T4 ×2** with Internet enabled.

This notebook audits the remaining large-model numerical question for:
- Qwen3-8B at intervention layer 14
- Mistral-7B-v0.3 at intervention layer 12
- 16 frozen audit displacements per model
- sequence length 64
- centered finite difference at ε=0.075
- half-step ε/2
- Richardson extrapolation
- exact FP32 JVP of the downstream suffix

Paper wording if successful: **“exact FP32 downstream-suffix JVP from the same cached natural receiver states.”**
Do not call it full-model FP32 JVP; the upstream natural states are reproduced in FP16.

If the notebook fails, keep the full traceback and do not change the layer, sequence length, precision, or ε.


In [ ]:
!pip -q install "transformers>=4.53.0" "accelerate>=1.2.0" "datasets>=3.0.0" scipy pandas tqdm


In [ ]:
import os, gc, json, math, random, hashlib, zipfile, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from tqdm.auto import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from accelerate import dispatch_model

warnings.filterwarnings("ignore")
assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 2, "Select Kaggle GPU T4 x2"
for i in range(2):
    p=torch.cuda.get_device_properties(i)
    print(i, torch.cuda.get_device_name(i), f"{p.total_memory/2**30:.1f} GiB")
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
try: torch.set_float32_matmul_precision('highest')
except Exception: pass


In [ ]:
SEED=26091016
SEQ_LEN=64
D=96
R=8
JVP_N=16
FD_EPS=0.075
MODEL_SPECS={
 'qwen3_8b': {'model_id':'Qwen/Qwen3-8B','layer':14},
 'mistral_7b': {'model_id':'mistralai/Mistral-7B-v0.3','layer':12},
}
OUTDIR=Path('/kaggle/working/fp32_suffix_exact_jvp_audit'); OUTDIR.mkdir(parents=True,exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)


In [ ]:
def sha256_text(s): return hashlib.sha256(s.encode('utf-8',errors='ignore')).hexdigest()
def load_contexts(n=240):
    ds=load_dataset('allenai/c4','en',split='validation',streaming=True)
    out=[]
    for ex in ds:
        text=(ex.get('text') or '').strip()
        if len(text)<500: continue
        h=sha256_text(text)
        if int(h[:8],16)%1000>=80: continue
        out.append({'context_id':h[:20],'text':text})
        if len(out)>=n: break
    return out
RAW_CONTEXTS=load_contexts(max(2*D+R+40,240))
print('deterministic contexts',len(RAW_CONTEXTS))
def encode_exact(tok,text,n=SEQ_LEN):
    ids=tok(text,add_special_tokens=False,truncation=False)['input_ids']
    if len(ids)<n: return None
    return torch.tensor(ids[:n],dtype=torch.long)
def tokenized_rows(tok,raw,need):
    rows=[]
    for r in raw:
        ids=encode_exact(tok,r['text'])
        if ids is None: continue
        rows.append({**r,'input_ids':ids})
        if len(rows)>=need: break
    if len(rows)<need: raise RuntimeError(f'Need {need}, got {len(rows)}')
    return rows
def layers(model): return model.model.layers
def mlp(model,idx): return layers(model)[idx].mlp
def fisher_energy(v,p):
    pv=(p*v).sum(dim=-1)
    return 0.5*((p*v.square()).sum(dim=-1)-pv.square())
def cos_sim(a,b): return float(torch.dot(a,b)/(a.norm()*b.norm()).clamp_min(1e-30))
def safe_spearman(x,y):
    x,y=np.asarray(x,float),np.asarray(y,float)
    return float(spearmanr(x,y).statistic) if len(x)>=3 and np.std(x)>0 and np.std(y)>0 else np.nan
def clear_cuda():
    gc.collect(); torch.cuda.empty_cache()
    for i in range(2):
        try: torch.cuda.reset_peak_memory_stats(i)
        except Exception: pass
def print_mem(tag):
    print(tag,' | '.join([f'gpu{i}: alloc={torch.cuda.memory_allocated(i)/2**30:.2f}G peak={torch.cuda.max_memory_allocated(i)/2**30:.2f}G' for i in range(2)]))


In [ ]:
@torch.inference_mode()
def capture_target(model,ids,layer_idx,want_mlp=True,want_post_layer=False):
    mb,lb=[],[]; hs=[]
    if want_mlp:
        hs.append(mlp(model,layer_idx).register_forward_hook(lambda m,inp,out: mb.append(out[:,-1,:].detach().float().cpu())))
    if want_post_layer:
        def hook(m,inp,out):
            h=out[0] if isinstance(out,(tuple,list)) else out
            lb.append(h.detach().float().cpu())
        hs.append(layers(model)[layer_idx].register_forward_hook(hook))
    try:
        dev=next(model.parameters()).device
        _=model(input_ids=ids[None,:].to(dev),use_cache=False,return_dict=True)
    finally:
        for h in hs: h.remove()
    return (mb[-1][0] if mb else None),(lb[-1] if lb else None)


In [ ]:
def make_pruned_suffix_fp32(model_id,intervention_layer):
    print('Loading CPU FP16 copy for pruning...')
    m=AutoModelForCausalLM.from_pretrained(model_id,torch_dtype=torch.float16,device_map={'':'cpu'},low_cpu_mem_usage=True,attn_implementation='eager')
    m.eval()
    kept=list(m.model.layers[intervention_layer+1:])
    m.model.layers=torch.nn.ModuleList(kept)
    try: m.model.embed_tokens=None
    except Exception: pass
    gc.collect()
    m=m.float()
    n=len(m.model.layers); split=n//2
    device_map={}
    for i in range(n): device_map[f'model.layers.{i}']=0 if i<split else 1
    device_map['model.norm']=1; device_map['lm_head']=1
    m=dispatch_model(m,device_map=device_map)
    for p in m.parameters(): p.requires_grad_(False)
    clear_cuda(); print('retained suffix layers',n,'split',split,'/',n-split); print_mem('FP32 suffix')
    return m


In [ ]:
def first_param_device(model): return next(p for p in model.parameters() if p is not None).device
def suffix_logits(model,post_hidden_cpu,delta_cpu,alpha):
    dev=first_param_device(model)
    h=post_hidden_cpu.to(dev,dtype=torch.float32); d=delta_cpu.to(dev,dtype=torch.float32)
    mask=torch.zeros_like(h); mask[:,-1,:]=d
    x=h+alpha.to(dev)*mask
    return model(inputs_embeds=x,use_cache=False,return_dict=True).logits[0,-1,:]
def exact_suffix_jvp(model,post_hidden_cpu,delta_cpu):
    dev=first_param_device(model)
    alpha=torch.zeros((),device=dev,dtype=torch.float32,requires_grad=True)
    tangent=torch.ones_like(alpha)
    def f(a): return suffix_logits(model,post_hidden_cpu,delta_cpu,a)
    y,v=torch.autograd.functional.jvp(f,alpha,tangent,create_graph=False,strict=False)
    return y.detach().cpu(),v.detach().cpu()
@torch.inference_mode()
def suffix_fd(model,post_hidden_cpu,delta_cpu,eps):
    dev=first_param_device(model)
    ap=torch.tensor(float(eps),device=dev,dtype=torch.float32); am=-ap
    zp=suffix_logits(model,post_hidden_cpu,delta_cpu,ap); zm=suffix_logits(model,post_hidden_cpu,delta_cpu,am)
    return ((zp-zm)/(2*eps)).detach().cpu()


In [ ]:
all_rows=[]
for model_key,spec in MODEL_SPECS.items():
    print('\n'+'='*90); print(model_key,spec); print('='*90)
    tok=AutoTokenizer.from_pretrained(spec['model_id'],use_fast=True)
    if tok.pad_token_id is None: tok.pad_token_id=tok.eos_token_id
    model=AutoModelForCausalLM.from_pretrained(spec['model_id'],torch_dtype=torch.float16,device_map='balanced',low_cpu_mem_usage=True,attn_implementation='eager')
    model.eval(); [p.requires_grad_(False) for p in model.parameters()]
    print_mem('full FP16 model')
    rows=tokenized_rows(tok,RAW_CONTEXTS,2*D+R); hosts=rows[:D]; donors=rows[D:2*D]; receivers=rows[2*D:2*D+R]; receiver=receivers[0]
    host_u=[]; donor_u=[]
    for i in tqdm(range(JVP_N),desc=f'{model_key}: cache'):
        hu,_=capture_target(model,hosts[i]['input_ids'],spec['layer'],True,False)
        du,_=capture_target(model,donors[i]['input_ids'],spec['layer'],True,False)
        host_u.append(hu); donor_u.append(du)
    _,receiver_post=capture_target(model,receiver['input_ids'],spec['layer'],False,True)
    deltas=[(donor_u[i]-host_u[i]).float() for i in range(JVP_N)]
    del model; clear_cuda()
    suffix=make_pruned_suffix_fp32(spec['model_id'],spec['layer'])
    rows_out=[]
    try:
        for i in tqdm(range(JVP_N),desc=f'{model_key}: JVP'):
            delta=deltas[i]
            z0,v_exact=exact_suffix_jvp(suffix,receiver_post,delta); p=torch.softmax(z0.float(),dim=-1)
            v075=suffix_fd(suffix,receiver_post,delta,FD_EPS); vhalf=suffix_fd(suffix,receiver_post,delta,FD_EPS/2); vrich=(4*vhalf-v075)/3
            q_exact=float(fisher_energy(v_exact[None],p[None])[0]); q075=float(fisher_energy(v075[None],p[None])[0]); qhalf=float(fisher_energy(vhalf[None],p[None])[0]); qrich=float(fisher_energy(vrich[None],p[None])[0])
            rec={'model':model_key,'pair_id':i,'host_context_id':hosts[i]['context_id'],'donor_context_id':donors[i]['context_id'],'receiver_context_id':receiver['context_id'],'delta_norm':float(delta.norm()),'cos_fd075_exact':cos_sim(v075,v_exact),'cos_half_exact':cos_sim(vhalf,v_exact),'cos_rich_exact':cos_sim(vrich,v_exact),'rel_l2_fd075_exact':float((v075-v_exact).norm()/v_exact.norm().clamp_min(1e-30)),'rel_l2_half_exact':float((vhalf-v_exact).norm()/v_exact.norm().clamp_min(1e-30)),'rel_l2_rich_exact':float((vrich-v_exact).norm()/v_exact.norm().clamp_min(1e-30)),'q_exact':q_exact,'q_fd075':q075,'q_half':qhalf,'q_richardson':qrich,'rel_qerr_fd075':abs(q075-q_exact)/max(abs(q_exact),1e-30),'rel_qerr_half':abs(qhalf-q_exact)/max(abs(q_exact),1e-30),'rel_qerr_rich':abs(qrich-q_exact)/max(abs(q_exact),1e-30)}
            rows_out.append(rec); all_rows.append(rec)
            pd.DataFrame(rows_out).to_csv(OUTDIR/f'{model_key}_FP32_suffix_exact_JVP.csv',index=False)
            pd.DataFrame(all_rows).to_csv(OUTDIR/'ALL_MODELS_FP32_suffix_exact_JVP.csv',index=False)
            print(model_key,i,rec['cos_fd075_exact'],rec['rel_qerr_fd075'])
    finally:
        del suffix,host_u,donor_u,deltas,receiver_post; clear_cuda()
all_df=pd.DataFrame(all_rows); display(all_df.head())


In [ ]:
summary=[]
for model,g in all_df.groupby('model'):
    summary.append({'model':model,'N':len(g),'median_cos_fd075_exact':float(g.cos_fd075_exact.median()),'median_cos_half_exact':float(g.cos_half_exact.median()),'median_cos_rich_exact':float(g.cos_rich_exact.median()),'median_rel_l2_fd075_exact':float(g.rel_l2_fd075_exact.median()),'median_rel_qerr_fd075':float(g.rel_qerr_fd075.median()),'median_rel_qerr_half':float(g.rel_qerr_half.median()),'median_rel_qerr_rich':float(g.rel_qerr_rich.median()),'rho_Qfd075_Qexact':safe_spearman(g.q_fd075,g.q_exact),'rho_Qhalf_Qexact':safe_spearman(g.q_half,g.q_exact),'rho_Qrich_Qexact':safe_spearman(g.q_richardson,g.q_exact)})
summary=pd.DataFrame(summary); summary.to_csv(OUTDIR/'FP32_suffix_exact_JVP_summary.csv',index=False); display(summary)
lines=[]
for _,r in summary.iterrows():
    pretty='Qwen3-8B' if r.model=='qwen3_8b' else 'Mistral-7B'
    lines.append(f"{pretty}: N={int(r.N)}, median cosine(FD eps=.075, exact FP32 suffix JVP)={r.median_cos_fd075_exact:.4f}; Spearman(Q_FD,Q_exact)={r.rho_Qfd075_Qexact:.4f}; median relative Q error={r.median_rel_qerr_fd075:.4f}.")
txt='\n'.join(lines); (OUTDIR/'PAPER_READY_SUMMARY.txt').write_text(txt); print(txt)


In [ ]:
metadata={'audit':'exact FP32 downstream-suffix JVP','prefix_state_generation':'FP16, model-parallel across 2x T4','suffix_precision':'FP32','tf32_disabled':True,'seq_len':SEQ_LEN,'epsilon':FD_EPS,'pairs_per_model':JVP_N,'qwen_intervention_layer':14,'mistral_intervention_layer':12,'seed':SEED}
(OUTDIR/'AUDIT_METADATA.json').write_text(json.dumps(metadata,indent=2))
zip_path=Path('/kaggle/working/FP32_SUFFIX_EXACT_JVP_AUDIT_RESULTS.zip')
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUTDIR.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(OUTDIR))
print('DONE — download:',zip_path)
